# 模块1：项目背景与数据概览教学
---
## 🎯 学习目标
学完本模块你将能够：
- 理解阿里天池推荐算法赛题的业务背景和评估指标
- 熟练使用pandas完成数据加载、基础统计、缺失值/重复值检查
- 掌握时序数据的基本处理方法
- 能够独立完成任何结构化数据集的初步探索分析

---
## 📚 知识点讲解：赛题与业务背景

### 1. 赛题介绍
本次是阿里天池新人实战赛之【复购预测】赛道：
- **业务场景**：电商平台用户行为数据分析，预测用户在未来一段时间内是否会复购某类商品
- **数据来源**：阿里巴巴提供的真实脱敏用户行为数据（2014年11月18日-2014年12月18日共30天）
- **任务目标**：预测用户在接下来7天内是否会购买给定子集的商品

### 2. 数据说明
我们有两张核心表：
| 表名 | 字段 | 说明 |
|------|------|------|
| 用户行为表 | user_id | 用户唯一ID |
| | item_id | 商品唯一ID |
| | behavior_type | 行为类型：1=浏览，2=收藏，3=加购，4=购买 |
| | user_geohash | 用户地理位置（脱敏） |
| | item_category | 商品所属类目 |
| | time | 行为发生时间 |
| 商品子集表 | item_id | 商品唯一ID |
| | item_geohash | 商品位置（脱敏） |
| | item_category | 商品所属类目 |

### 3. 评估指标
比赛使用**F1-score**作为最终评估指标：
$$F1 = 2 * \frac{Precision * Recall}{Precision + Recall}$$
- Precision（精确率）：预测为复购的用户中，实际真的复购的比例
- Recall（召回率）：实际复购的用户中，被我们正确预测出来的比例
- F1是两者的调和平均，能够平衡精确率和召回率的表现

---
## 💻 代码逐行拆解：数据加载与概览


In [ ]:
# 1. 导入需要的库
import pandas as pd
from IPython.display import display

**参数详解：**
- `import pandas as pd`：导入pandas数据分析库，约定俗成缩写为pd
- `from IPython.display import display`：导入display函数，用于更美观地打印DataFrame


In [ ]:
# 2. 加载数据
df_user = pd.read_csv(
    "F:/数据分析/project/天池/新人实战/fresh_comp_offline/tianchi_fresh_comp_train_user.csv"
)
df_item = pd.read_csv(
    "f:/数据分析/project/天池/新人实战/fresh_comp_offline/tianchi_fresh_comp_train_item.csv"
)

**参数详解：**
- `pd.read_csv()`：pandas读取csv文件的核心函数
- 第一个参数是文件的绝对路径，确保路径正确才能成功读取
- 常用可选参数：
  - `sep=','`：指定分隔符，默认是逗号
  - `header=0`：指定第几行作为列名，默认第一行
  - `encoding='utf-8'`：指定文件编码，遇到中文乱码时可以尝试'gbk'


In [ ]:
# 3. 查看用户行为表基础信息
display(df_user.shape)  # 查看数据表的行数和列数
display(df_user.head())  # 查看前5行数据
display(df_user.dtypes)  # 查看每列的数据类型

**知识点：**
- `shape`属性返回一个元组(行数, 列数)，可以快速了解数据规模
- `head(n)`方法返回前n行数据，默认n=5，用于快速预览数据格式
- `dtypes`属性返回每列的数据类型，常见类型：int64(整数)、float64(浮点数)、object(字符串/混合类型)

**输出说明：**
- 我们可以看到用户行为表有2300多万行，6个字段，是非常大的数据集
- 行为类型是整数，时间是字符串类型，后续需要转换为时间类型


In [ ]:
# 4. 查看商品子集表基础信息
display(df_item.shape)
display(df_item.head())
display(df_item.dtypes)

**输出说明：**
- 商品子集表有62万多行，3个字段，是我们需要预测是否被购买的商品集合


---
## 💻 代码逐行拆解：数据质量检查

### 1. 缺失值检查

In [ ]:
user_missing_summary = df_user.isnull().sum()  # 统计每列缺失值的数量
user_missing_ratio = df_user.isnull().mean() * 100  # 统计每列缺失值的占比
display(user_missing_summary)
display(user_missing_ratio)

**参数详解：**
- `isnull()`方法：返回一个和原DataFrame同样大小的布尔值矩阵，缺失值位置为True，否则为False
- `sum()`方法：对布尔值求和，True会被当作1，False当作0，得到每列缺失值总数
- `mean()`方法：求平均值，乘以100得到缺失值的百分比

**结果解读：**
- user_geohash列缺失率高达68%，这个字段我们后续可以考虑删除或者特殊处理
- 其他字段没有缺失值，质量很好


In [ ]:
# 检查商品表缺失值
item_missing_summary = df_item.isnull().sum()
item_missing_ratio = df_item.isnull().mean() * 100
display(item_missing_summary)
display(item_missing_ratio)

**结果解读：**
- item_geohash列缺失率67%，同样可以考虑后续删除


### 2. 重复值检查

In [ ]:
# 完全重复检查：整行数据完全一样
user_duplicates = df_user.duplicated().sum()
print(f"完全重复的行数: {user_duplicates}")

item_duplicates = df_item.duplicated().sum()
print(f"商品表完全重复的行数: {item_duplicates}")

**参数详解：**
- `duplicated()`方法：返回布尔值序列，重复的行返回True，否则False
- 可选参数`subset=[列名列表]`：可以指定按哪些列判断重复，默认按所有列

**结果解读：**
- 用户行为表有780多万行完全重复的数据，后续需要去重
- 商品表没有重复数据


In [ ]:
# 业务逻辑重复检查：同一个用户在同一个时间对同一个商品做了同一个行为
key_duplicates = df_user.duplicated(
    subset=["user_id", "item_id", "behavior_type", "time"]
).sum()
print(f"关键业务逻辑重复的行数: {key_duplicates}")

**结果解读：**
- 按业务关键字段判断，有1150多万行重复数据，这些重复数据没有业务价值，必须去重


### 3. 字段唯一性检查

In [ ]:
# 检查行为类型的唯一值和分布
print("行为类型唯一值:", df_user["behavior_type"].unique())
print("行为类型分布:\n", df_user["behavior_type"].value_counts())

# 检查核心维度的数量
print(f"独立用户数: {df_user['user_id'].nunique()}")
print(f"独立商品数: {df_user['item_id'].nunique()}")
print(f"商品类目数: {df_user['item_category'].nunique()}")

**参数详解：**
- `unique()`方法：返回列中所有不重复的值，用于了解字段的取值范围
- `value_counts()`方法：统计每个取值出现的次数，默认按降序排列
- `nunique()`方法：返回列中不重复值的数量，相当于len(unique())

**结果解读：**
- 行为类型确实只有1/2/3/4四种，和我们的业务定义一致
- 浏览行为占绝大多数(94%)，购买行为仅占1%，这符合真实电商场景的漏斗分布
- 共有2万用户，475万商品，9557个类目，数据规模很大


---
## 💻 代码逐行拆解：时间字段处理


In [ ]:
# 1. 字符串时间转换为datetime类型
df_user["time"] = pd.to_datetime(df_user["time"])

# 检查时间范围
print(f"最早时间：{df_user['time'].min()}")
print(f"最晚时间：{df_user['time'].max()}")

**参数详解：**
- `pd.to_datetime()`：将字符串格式的时间转换为pandas的datetime64类型，方便后续做时间运算
- `.min()`和`.max()`方法可以快速获取时间列的最小和最大值，了解数据的时间范围

**结果解读：**
- 数据覆盖了2014-11-18到2014-12-18整整30天的用户行为，和赛题说明一致


In [ ]:
# 2. 从时间中提取小时、天、日期字段
df_user["hour"] = df_user["time"].dt.hour  # 提取小时(0-23)
df_user["day"] = df_user["time"].dt.day    # 提取日期(1-31)
df_user["date"] = df_user["time"].dt.date  # 提取日期部分(年-月-日)

display(df_user.head())

**知识点：**
- pandas的datetime类型通过`.dt`访问器可以方便地提取各种时间属性：
  - `.dt.year`/`.dt.month`/`.dt.day`：年/月/日
  - `.dt.hour`/`.dt.minute`/`.dt.second`：时/分/秒
  - `.dt.weekday`：星期几(0=周一, 6=周日)
  - `.dt.date`：日期部分，返回datetime.date类型


In [ ]:
# 3. 检查时间连续性
all_dates = pd.date_range(
    start=df_user["time"].min(), end=df_user["time"].max(), freq="D"
).date
actual_dates = set(df_user["date"].unique())
missing_dates = set(all_dates) - actual_dates

if not missing_dates:
    print("日期是连续的")
else:
    print(f"发现不连续的日期{list(missing_dates)}")

**参数详解：**
- `pd.date_range()`：生成一个连续的时间序列
  - `start`：起始时间
  - `end`：结束时间
  - `freq`：时间频率，'D'=天，'h'=小时，'M'=月

**作用：**
- 检查数据是否有日期缺失，如果有缺失需要了解缺失原因，避免影响后续分析


---
## 💻 代码逐行拆解：表关联验证


In [ ]:
# 获取两个表的商品集合
user_items = set(df_user['item_id'].unique())
item_subset = set(df_item['item_id'].unique())

# 计算差集
items_not_in_subset = user_items - item_subset  # 在用户行为中但不在预测子集的商品
items_no_behavior = item_subset - user_items    # 在预测子集中但没有任何行为的商品

print(f"用户行为表中唯一商品数: {len(user_items)}")
print(f"商品子集表中唯一商品数: {len(item_subset)}")
print(f"用户行为中有但不在预测子集的商品数量: {len(items_not_in_subset)}")
print(f"预测子集中但没有任何行为的商品数量: {len(items_no_behavior)}")

**结果解读：**
- 用户行为表中有475万商品，但我们只需要预测其中42万商品的复购情况
- 预测子集中的所有商品都有用户行为数据，没有冷启动问题
- 后续建模时我们只需要关注这42万商品的用户行为即可，可以大大减少数据量


---
## 🎯 实战例题

请你独立完成以下练习：
1. 统计用户行为表中，每个行为类型的用户数（不是行为次数）
2. 找出有购买行为的用户占总用户数的比例
3. 提取时间列的星期几字段，统计每天不同星期的总行为量
4. 统计12月12日（双十二）当天的购买量比平日平均购买量高多少倍


---
## ✅ 小测验

### 选择题
1. 以下哪个pandas方法可以用来统计每列的缺失值数量？
   A. df.isnull().count()
   B. df.isnull().sum()
   C. df.missing().sum()
   D. df.count_missing()

2. pd.to_datetime()函数的作用是？
   A. 将datetime类型转换为字符串
   B. 将字符串转换为datetime类型
   C. 提取时间中的日期部分
   D. 计算两个时间的差值

3. 本次赛题的评估指标是F1-score，它是哪两个指标的调和平均？
   A. 准确率和召回率
   B. 精确率和召回率
   C. 精确率和准确率
   D. 召回率和AUC

### 简答题
1. 为什么在数据分析第一步要做缺失值和重复值检查？
2. 用户行为表中user_geohash字段缺失率68%，你会怎么处理这个字段？


---
## 📝 重点总结

本模块核心知识点：
1. **数据探索通用流程**：加载数据→基础概览→数据质量检查（缺失/重复/字段取值）→时间处理→关联验证
2. **pandas核心方法**：
   - 数据加载：pd.read_csv()
   - 基础概览：shape/head()/dtypes
   - 缺失值：isnull().sum()
   - 重复值：duplicated().sum()
   - 字段统计：unique()/value_counts()/nunique()
   - 时间处理：pd.to_datetime()/.dt访问器/pd.date_range()
3. **业务理解**：电商用户行为漏斗（浏览>加购>收藏>购买）、复购预测的业务价值

下一个模块我们将学习如何对这些数据进行可视化分析，挖掘用户行为的规律。